# 3.3 Linear Regression

**Core series, class 8.** Diabetes and high blood pressure often go together. In this class the outcome is a **number**, **systolic blood pressure** (SBP), so we use **linear regression**: how much higher, on average, is SBP for each year of age, each BMI unit, and in people with type 2 diabetes, holding the others constant?

### By the end of this notebook you'll be able to

1. Explain a linear regression **coefficient**: the average change in the outcome per unit of a predictor
2. Fit and read models with `lm()`, `broom::tidy()`, and `broom::glance()`
3. Compare models with **R²**, **adjusted R²**, and **AIC**
4. Check a model's **residuals**

### 🔎 Analysis at a glance

| | |
|---|---|
| ❓ **Question** | Is type 2 diabetes linked to higher systolic blood pressure, after accounting for age, sex, BMI, and LDL? |
| 🧪 **The analysis** | **Linear regression**: SBP ~ age + sex + BMI + LDL (+ type 2 diabetes), built up step by step, on the same people |
| 🎯 **Why** | SBP is a **number**, so linear regression; adjusting separates diabetes from age and weight, which also raise blood pressure |
| ✅ **What we conclude** | The adjusted difference in SBP (mm Hg) between people with and without type 2 diabetes, and how much each model explains |
| ⚠️ **What it can't tell us** | Cause, or untreated blood pressure: many people take blood pressure medicine, and our value is each person's **latest** reading |
| 🔗 **Connections** | Uses `final_analysis.csv`; the logistic version is **3.2**; the same kind of model is used for gene activity in **4.5** |

| | |
|---|---|
| 📂 **Uses** | `final_analysis.csv` (from **2.3**) |
| 💾 **Saves** | nothing |

### Class plan

| Part | Minutes |
|---|---|
| 1. Key concepts | 15 |
| 2. Load and look at the outcome | 10 |
| 3. One predictor | 15 |
| 4. Multivariable models | 20 |
| 5. Check the residuals | 10 |
| Wrap-up | 5 |

### ⏱️ Time and 💰 cost

| | |
|---|---|
| ⏱️ **Class time** | 75 minutes |
| ⏱️ **Run time** | under **1 minute** to run every cell |
| 💰 **Data cost** | **$0**: no BigQuery queries |
| 💰 **App cost** | about **$0.27 an hour** while the app is running. The app's disk costs about **$0.70 a day**, even while the app is stopped |
| **Estimated total** | about **$0.35** per student for this class (app time only) |

> ⚠️ **Keep All of Us data in the Workbench.** Plots with one dot per person stay in the notebook: don't screenshot them. Clear outputs before you share or commit this notebook (**Edit → Clear Outputs of All Cells**).

## Part 1. Key concepts

### The model

$$\text{SBP} = \beta_0 + \beta_1 \times \text{age} + \beta_2 \times \text{BMI} + \dots + \text{error}$$

- **β₀ (intercept):** the predicted SBP when every predictor is 0 (often not meaningful on its own).
- **β₁, β₂, … (coefficients):** the average change in SBP (mm Hg) **per unit** of each predictor, holding the others constant. For a yes/no predictor like type 2 diabetes, it's the average **difference** between the groups.
- **Error (residual):** how far each person's SBP is from the model's prediction.

### How well does it fit?

| Measure | Means | Better is |
|---|---|---|
| **R²** | the share of the variation in SBP the model explains (0 to 1) | higher |
| **Adjusted R²** | R², with a penalty for each extra predictor | higher |
| **AIC** | fit vs complexity (only comparable on the same data) | lower |

Blood pressure varies a lot from reading to reading, so even good models have a **low R²** here. A low R² doesn't make a coefficient wrong: the model can still estimate an average difference well.

### Assumptions to check

Linear regression assumes the residuals are roughly **centred on 0** with a **similar spread** everywhere. We check with a **residual plot** (Part 5).

## Part 2. Load and look at the outcome

In [ ]:
source("../aou_setup.R")  # settings, packages, and helper functions
library(broom)

fa <- prepare_analysis_data() %>%
  filter(sex_at_birth %in% c("Female", "Male")) %>%
  mutate(sex_at_birth = droplevels(sex_at_birth),
         age10 = age / 10, BMI5 = BMI / 5, LDL10 = LDL / 10)

summary(fa$SBP)

ggplot(fa, aes(SBP)) +
  geom_histogram(bins = 40, fill = "#3182bd") +
  theme_bw() +
  labs(x = "Systolic blood pressure, latest reading (mm Hg)", y = "People")

**🔍 Read the output:** most values fall between about 100 and 160 mm Hg, in a hump with a longer tail to the right. Normal is below 120; 130 or more is high blood pressure in US guidelines.

## Part 3. One predictor

Age is the strongest single predictor of blood pressure.

In [ ]:
m_age <- lm(SBP ~ age10, data = fa)
tidy(m_age, conf.int = TRUE)

ggplot(fa, aes(age, SBP)) +
  geom_point(alpha = 0.05) +
  geom_smooth(method = "lm", formula = y ~ x, colour = "#d95f02") +
  theme_bw() +
  labs(x = "Age (years)", y = "Systolic blood pressure (mm Hg)")

**🔍 Read the output:**

- The `age10` row's `estimate`: the average increase in SBP **per 10 years** of age, in mm Hg.
- The plot: each dot is a person (faint, so you can see where they cluster); the line is the model. The scatter around the line is large: age explains only part of the variation.

### ✍️ Your turn

Fit SBP ~ **t2d** on its own. How much higher is SBP in cases, before any adjustment?

<details>
<summary><b>Show answer</b></summary>

```r
lm(SBP ~ t2d, data = fa) %>% tidy(conf.int = TRUE)
```

The `t2dYes` estimate is the **crude** difference in mean SBP between cases and controls, in mm Hg. Part 4 adjusts it.

</details>

In [ ]:
# Your code here

## Part 4. Multivariable models

As in 3.2, all models use the **same people**: those with every variable.

In [ ]:
model_data <- fa %>% drop_na(SBP, age10, sex_at_birth, BMI5, LDL10, t2d)
nrow(model_data)

m1 <- lm(SBP ~ age10 + sex_at_birth,                       data = model_data)
m2 <- lm(SBP ~ age10 + sex_at_birth + BMI5 + LDL10,        data = model_data)
m3 <- lm(SBP ~ age10 + sex_at_birth + BMI5 + LDL10 + t2d,  data = model_data)

bind_rows("1. Age + sex" = glance(m1), "2. + BMI, LDL" = glance(m2), "3. + type 2 diabetes" = glance(m3), .id = "model") %>%
  select(model, nobs, r.squared, adj.r.squared, AIC) %>%
  mutate(across(c(r.squared, adj.r.squared), ~ round(.x, 3)))

tidy(m3, conf.int = TRUE) %>%
  mutate(across(c(estimate, conf.low, conf.high), ~ round(.x, 2)))

**🔍 Read the output:**

- **The first table:** R² rises a little with each model, and AIC falls. Even the best model explains only a modest share of the variation (blood pressure is very variable).
- **The coefficients of model 3:** each `estimate` is in **mm Hg**:
  - `age10`: per 10 years of age
  - `sex_at_birthMale`: men compared with women
  - `BMI5`: per 5 BMI units
  - `t2dYes`: the **adjusted** difference between people with and without type 2 diabetes, at the same age, sex, BMI, and LDL

Compare `t2dYes` with the crude difference from the exercise above: does adjusting make it smaller?

**💬 Discuss:** many people with diabetes take blood pressure medicines (ACE inhibitors, for example, also protect the kidneys). How would that affect the diabetes coefficient? (It would make it **smaller** than the untreated difference: treatment hides part of the link.)

## Part 5. Check the residuals

A **residual** is each person's actual SBP minus the model's prediction. `augment()` adds predictions (`.fitted`) and residuals (`.resid`) to the data.

In [ ]:
augment(m3) %>%
  ggplot(aes(.fitted, .resid)) +
  geom_point(alpha = 0.05) +
  geom_hline(yintercept = 0, colour = "#d95f02") +
  geom_smooth(method = "loess", formula = y ~ x, se = FALSE, colour = "black") +
  theme_bw() +
  labs(x = "Predicted SBP (mm Hg)", y = "Residual (actual - predicted)")

**🔍 Read the output:** the cloud should be centred on the orange line at 0, with the black trend line close to flat and a similar spread from left to right. That's roughly what we see: the assumptions are reasonable. A curved trend would suggest a non-linear relationship (e.g. age's effect speeding up); a funnel shape would suggest unequal spread.

## ⚠️ Common errors

| Message | Means | Fix |
|---|---|---|
| Different `nobs` between models | Each model dropped different missing rows | Fit all models on one `model_data` |
| *`geom_smooth()` using formula …* | Just a message | Give `formula = y ~ x` to silence it |
| A huge plot that takes long to draw | Thousands of points | `alpha` helps; or plot a sample with `slice_sample()` |

## ✅ Key takeaways

- Linear regression is for **numeric** outcomes; each coefficient is an average change **per unit**
- **Adjusted** coefficients hold the other predictors constant
- A low **R²** doesn't make a coefficient wrong; compare models with AIC on the same people
- **Check the residuals**
- **Treatment** (blood pressure medicine) can hide a link in observational data

**Next:** **3.4 survival_analysis**: time from type 2 diabetes to chronic kidney disease.

**Before you share or commit this notebook:** clear all outputs (**Edit → Clear Outputs of All Cells**).